In [1]:
from utils import *

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = "cuda:1"

/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def plotMetrics(metrics):
    nodeX, nodeY = np.zeros([len(metrics), 4]), np.zeros([len(metrics), 4])
    precisionBox, recallBox, f1Box = np.zeros([len(metrics), config.future, 4]), np.zeros([len(metrics), config.future, 4]), np.zeros([len(metrics), config.future, 4])
    for i, name in enumerate(metrics):
        nodeX[i, :] = metrics[name]["nodes"].cpu().numpy()

        tp = metrics[name]["tp"]
        fp = metrics[name]["fp"]
        fn = metrics[name]["fn"]

        recall = tp / (tp + fn)
        precision = tp / (tp + fp)

        f1 = 2 * recall * precision / (recall + precision)

        nodeY[i, :] = torch.mean(f1, dim=0).cpu().numpy()

        precisionBox[i] = precision.cpu().numpy()
        recallBox[i] = recall.cpu().numpy()
        f1Box[i] = f1.cpu().numpy()

    plt.figure(figsize=(20, 12))

    labels = ["1 Year Return Period", "2 Year Return Period", "5 Year Return Period", "10 Year Return Period"]
    colors = ["blue", "green", "yellow", "orange"]
    for i in range(4):
        plt.subplot(2, 4, i + 1)
        plt.title(labels[i])
        currentX = nodeX[:, i]
        currentY = nodeY[:, i]
        mask = ~np.isnan(currentY)
        currentX, currentY = currentX[mask], currentY[mask]
        plt.scatter(currentX, currentY, alpha=0.3, c=colors[i])
        plt.ylim(0, 1)

        plt.grid()
        plt.xlabel("Total Upstream Basin Nodes")
        plt.ylabel("F1 Score")

    for i in range(4):
        plt.subplot(2, 4, i + 5)
        plt.title(labels[i])
        currentF1 = f1Box[:, :, i].T
        currentF1 = [box[~np.isnan(box)] for box in currentF1]
        plt.boxplot(currentF1)
        plt.ylim(0, 1)

        plt.grid()
        plt.xlabel("Forecast Horizon")
        plt.ylabel("F1 Score")

    plt.show()

    # [basins, 1], [basins, timesteps]
    nodeErrorX, nodeErrorY = np.array([metrics[name]["nodes"].cpu().numpy() for name in metrics]), np.array([metrics[name]["mae"].cpu().numpy() for name in metrics])
    plt.figure(figsize=(20, 6))
    for i in range(config.future):
        plt.subplot(1, config.future, i + 1)
        plt.scatter(nodeErrorX, nodeErrorY[:, i])
        plt.title(f"NMAE @ Horizon {i + 1}")
        plt.xlabel("Graph Size (# of Nodes)")
        if i == 0:
            plt.ylabel("NMAE")
        plt.grid()
    plt.show()

    sanityY = np.array([metrics[name]["sanity"].cpu().numpy() for name in metrics])
    plt.figure(figsize=(20, 6))
    for i in range(config.future):
        plt.subplot(1, config.future, i + 1)
        plt.scatter(nodeErrorX, sanityY[:, i])
        plt.title(f"Sanity Check @ Horizon {i + 1}")
        plt.xlabel("Graph Size (# of Nodes)")
        if i == 0:
            plt.ylabel("NMAE")
        plt.grid()
    plt.show()

    kge = []
    for name in metrics:
        alpha = metrics[name]["corr"].meanX / metrics[name]["targetMean"]
        beta = math.sqrt(metrics[name]["corr"].varianceX) / metrics[name]["targetDev"]
        corr = metrics[name]["corr"].value()
        gaugeKGE = 1 - math.sqrt((corr - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)
        kge.append(gaugeKGE)
    kge = np.array(kge)
    
    nse = 1 - np.array([metrics[name]["nseNum"].cpu().numpy() / metrics[name]["nseDenom"].cpu().numpy() for name in metrics])
    nseCDF = np.array([np.sum(nse < (threshold / 1000)) / len(nse) for threshold in range(-1000, 1000)])
    kgeCDF = np.array([np.sum(kge < (threshold / 1000)) / len(kge) for threshold in range(-1000, 1000)])

    plt.figure(figsize=(12, 6))
    plt.plot(np.arange(-1000, 1000)/1000, nseCDF)
    plt.title("Cumulative Distribution of NSE")
    plt.xlabel("NSE")
    plt.ylabel("CDF")
    plt.grid()
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.plot(np.arange(-1000, 1000)/1000, kgeCDF)
    plt.title("Cumulative Distribution of NSE")
    plt.xlabel("KGE")
    plt.ylabel("CDF")
    plt.grid()
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.hist(np.clip(nse, -1, 1))
    plt.title("NSE Distribution")
    plt.xlabel("NSE")
    plt.ylabel("Frequency")
    plt.grid()
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.hist(np.clip(kge, -1, 1))
    plt.title("KGE Distribution")
    plt.xlabel("KGE")
    plt.ylabel("Frequency")
    plt.grid()
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.scatter(nodeErrorX, nse)
    plt.title("NSE by Nodes")
    plt.xlabel("Graph Size (# of Nodes)")
    plt.ylabel("NSE")
    plt.grid()
    plt.show()


In [3]:
class StreamingPearson:
    def __init__(self):
        self.n = 0
        self.meanX = 0
        self.meanY = 0
        self.varianceX = 0
        self.varianceY = 0
        self.covariance = 0

    def update(self, x, y):
        for i in range(x.shape[0]):
            self.n += 1
            weight = 1 / self.n
            dx = x[i].item() - self.meanX
            dy = y[i].item() - self.meanY
            self.meanX += weight * dx
            self.meanY += weight * dy
            self.varianceX = (1 - weight) * (self.varianceX + weight * dx * dx)
            self.varianceY = (1 - weight) * (self.varianceY + weight * dy * dy)
            self.covariance = (1 - weight) * (self.covariance + weight * dx * dy)

    def value(self):
        return self.covariance / math.sqrt(self.varianceX * self.varianceY)

In [ ]:
def evalModel(config, model, test):
    model.eval()
    model = model.to(device)

    totalMemory = torch.cuda.memory.mem_get_info()[1]

    metrics = {}

    progress = 0

    with torch.no_grad():
        targetMean = test.dataset.dataset.targetMean
        transform = test.dataset.dataset.transform

        try:
            for inputs, targets in test:
                inputs, targets = (inputs[0].to(device), inputs[1].to(device)), targets.to(device)
                history, future = targets.dischargeHistory, targets.dischargeFuture
                thresholds, means, deviations = targets.thresholds, targets.mean.unsqueeze(-1), targets.deviation.unsqueeze(-1)
                hindcast, forecast = model(inputs)

                meanPrediction = transform.backward(torch.mean(CMAL.sample(*forecast, 10000), dim=-1))
                future = transform.backward(future)
                rmse = torch.pow(meanPrediction - future, 2)

                nsePred = meanPrediction
                nseObs = future

                sanity = torch.abs(means - future)

                nseNumer = torch.sum(torch.pow(nseObs - nsePred, 2), dim=1)
                nseDenom = torch.sum(torch.pow(nseObs - means, 2), dim=1)
                # nseDenom = torch.sum(torch.pow(nseObs - targetMean, 2), dim=1)

                meanPrediction = meanPrediction.unsqueeze(-1)
                future = future.unsqueeze(-1)
                thresholds = thresholds.unsqueeze(1).expand(-1, config.future, -1)
                tp = (meanPrediction > thresholds).float() * (future > thresholds).float()
                fp = (meanPrediction > thresholds).float() * (future < thresholds).float()
                fn = (meanPrediction < thresholds).float() * (future > thresholds).float()

                past, _ = inputs
                for n, name in enumerate(past.grdcID):
                    if name not in metrics:
                        metrics[name] = {
                            "iter": 0,
                            "sanity": torch.zeros([config.future]).to(device),
                            "rmse": torch.zeros([config.future]).to(device),
                            "tp": torch.zeros([config.future, thresholds.shape[-1]]).to(device),
                            "fp": torch.zeros([config.future, thresholds.shape[-1]]).to(device),
                            "fn": torch.zeros([config.future, thresholds.shape[-1]]).to(device),
                            "nodes": past.nodes[n],
                            "nseNum": 0,
                            "nseDenom": 0,
                            "size": past.basinArea[n].item(),
                            "corr": StreamingPearson(),
                            "targetDev": targets.mean[n],
                            "targetMean": targets.deviation[n]
                        }

                    metrics[name]["sanity"] = (metrics[name]["sanity"] * metrics[name]["iter"] + sanity[n]) / (metrics[name]["iter"] + 1)
                    metrics[name]["rmse"] = (metrics[name]["rmse"] * metrics[name]["iter"] + rmse[n]) / (metrics[name]["iter"] + 1)
                    metrics[name]["tp"] += tp[n]
                    metrics[name]["fp"] += fp[n]
                    metrics[name]["fn"] += fn[n]
                    metrics[name]["nseNum"] += nseNumer[n]
                    metrics[name]["nseDenom"] += nseDenom[n]
                    metrics[name]["iter"] += 1
                    metrics[name]["corr"].update(meanPrediction[n], future[n])

                progress += 1
                print(f"\r{progress}/{len(test)} | {(progress / len(test)) * 100:.2f}% Complete | {100 * torch.cuda.memory_allocated() / totalMemory:.2f}% Memory Allocated", end="")

                # if progress % 4000 == 0:
                #     plotMetrics(metrics)

        except KeyboardInterrupt:
            pass

        plotMetrics(metrics)

    return metrics

In [ ]:
modelPaths = glob(os.path.join("checkpoints", "*/"))

for m, modelPath in enumerate(modelPaths):
    if os.path.exists(os.path.join(modelPath, "metrics.json")):
        continue

    if "floodHub" in modelPath:
        modelClass = FloodHub
        dataClass = FloodHubData
    else:
        modelClass = InundationBlockStation
        dataClass = InundationData
    
    config = Config().load(os.path.join(modelPath, "config.json"))
    model = modelClass(config)
    stateDict = torch.load(os.path.join(modelPath, "checkpoint.pt"), weights_only=True)
    model.load_state_dict(stateDict)
    dataset = dataClass(config)
    dataset.config.nodesPerBatch *= 4
    dataset.config.batchSize *= 4
    train, test = dataClass.split(dataset, trainSplit=config.dataSplit, seed=config.seed, shuffle=True)

    print("=" * 20, f"\nTESTING: {modelPath.split(" ")[-1]}", "\n" + "=" * 20)

    metrics = evalModel(config, model, test)

    for gauge in metrics:
        metrics[gauge]["correlation"] = metrics[gauge]["corr"].value()
        metrics[gauge]["predMean"] = metrics[gauge]["corr"].meanX
        metrics[gauge]["predDev"] = metrics[gauge]["corr"].varianceX
        metrics[gauge]["rmse"] = torch.sqrt(metrics[gauge]["rmse"])
        del metrics[gauge]["corr"]

    with open(os.path.join(modelPath, "metrics.json"), "w+") as file:
        for gauge in metrics:
            for metric in metrics[gauge]:
                if type(metrics[gauge][metric]) == torch.Tensor:
                    metrics[gauge][metric] = metrics[gauge][metric].cpu().numpy().tolist()
        json.dump(metrics, file, indent=4)

    del train, test, dataset, model

/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/utils/data/precompute.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grdcDF = pd.concat([newDF, grdcDF], ignore_index=True)
/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/utils/data/precompute.py:81: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  joined.to_file(joinedDFPath)
/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/venv/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Creating a 256th field, but some DBF readers might only support 255 fields
  ogr_write(
/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/venv/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'index_righ

Loading GeoPandas...
GeoPandas Loaded
2544/2544 GRDC files loaded (15.857000350952148, 5422.872275774429)))))
Loaded 9640/9640 ERA5 files
Total empty basins: 37
57622/57646 Basin Structures Appended to Graph
Upstream Basins Compiled | 1.0 | 17.43485477178423
Upstream Structures Compiled
Structure Tensors Complete
Index Mapping Complete
Static Input Scaling Complete
Total Useable Gauges: 804
Total Useable Basins: 9640
TESTING: floodHub/ 
104/1025 | 10.15% Complete | 0.01% Memory Allocated

/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/utils/data/precompute.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grdcDF = pd.concat([newDF, grdcDF], ignore_index=True)
/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/utils/data/precompute.py:81: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  joined.to_file(joinedDFPath)
/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/venv/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Creating a 256th field, but some DBF readers might only support 255 fields
  ogr_write(
/home/SGF.EDUBEAR.NET/dab4s/Inundation-Station/venv/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'index_righ

Loading GeoPandas...
GeoPandas Loaded
78/2544 GRDC files loaded (1.902469574666143, 430.0)00048828125))))